In [10]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.decomposition import PCA
import warnings
warnings.filterwarnings('ignore')

In [11]:
def preprocess_features(df):
    """Створює нові ознаки на основі базових аудіо-характеристик."""
    df_processed = df.copy()
    
    # 1. Energy_to_Acoustic_Ratio
    epsilon = 1e-5
    df_processed['Energy_to_Acoustic_Ratio'] = df_processed['energy'] / (df_processed['acoustic'] + epsilon)
    
    # 2. Is_Major (Витягуємо з колонки Key, наприклад "C Major" -> 1, "A Minor" -> 0)
    # Якщо колонка Key містить числа (Camelot), логіку можна адаптувати. 
    # Тут базова перевірка на наявність слова 'Major'
    if 'key' in df_processed.columns:
        df_processed['Is_Major'] = df_processed['key'].astype(str).apply(
            lambda x: 1 if 'major' in x.lower() else 0
        )
    else:
        df_processed['Is_Major'] = 0
        
    # 3. Tempo Bins (Кошики для BPM)
    bins = [0, 90, 110, 130, 300]
    labels = [0, 1, 2, 3]
    df_processed['Tempo_Bin'] = pd.cut(df_processed['bpm'], bins=bins, labels=labels).astype(float)
    
    # Заповнюємо можливі пропуски медіаною
    cols_to_fill = ['energy', 'valence', 'acoustic', 'dance', 'bpm', 'Energy_to_Acoustic_Ratio', 'Tempo_Bin']
    df_processed[cols_to_fill] = df_processed[cols_to_fill].fillna(df_processed[cols_to_fill].median())
    
    return df_processed

# Визначаємо базові ознаки, які підуть у модель
base_features = ['energy', 'valence', 'acoustic', 'dance', 'bpm', 'Energy_to_Acoustic_Ratio', 'Is_Major', 'Tempo_Bin']

In [ ]:
# Словник для зберігання артефактів кожної моделі
system_models = {}

configs = [
    {'file': 'weather.csv', 'target': 'weather_label', 'name': 'Weather'},
    {'file': 'time.csv', 'target': 'time_label', 'name': 'Time'},
    {'file': 'season.csv', 'target': 'season_label', 'name': 'Season'}
]

for config in configs:
    name = config['name']
    target_col = config['target']
    
    try:
        # Читаємо файл
        print(config['file'])
        df = pd.read_csv(config['file'])
        
        # Переводимо всі назви колонок у нижній регістр для зручності
        df.columns = df.columns.str.lower()
        
        # Застосовуємо інженерію ознак
        df_feat = preprocess_features(df)
        
        X_raw = df_feat[base_features]
        y_raw = df_feat[target_col]
        
        # PCA потребує масштабування (StandardScaler)
        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(X_raw)
        
        pca = PCA(n_components=2)
        pca_result = pca.fit_transform(X_scaled)
        
        # Додаємо PCA компоненти до фінального набору ознак
        X = X_raw.copy()
        X['PCA1'] = pca_result[:, 0]
        X['PCA2'] = pca_result[:, 1]
        
        # Кодування цільової змінної
        le = LabelEncoder()
        y = le.fit_transform(y_raw)
        
        # Тренування моделі XGBoost
        model = xgb.XGBClassifier(
            objective='multi:softprob', 
            num_class=len(le.classes_),
            eval_metric='mlogloss',
            max_depth=5,
            learning_rate=0.1,
            n_estimators=100,
            random_state=42
        )
        model.fit(X, y)
        
        # Зберігаємо все необхідне для інференсу
        system_models[name] = {
            'model': model,
            'le': le,
            'scaler': scaler,
            'pca': pca
        }
        
        print(f"Модель {name} успішно навчена! Класи: {le.classes_}")
        
    except FileNotFoundError:
        print(f"Файл {config['file']} не знайдено. Пропускаю...")

ParserError: Error tokenizing data. C error: Expected 24 fields in line 2251, saw 26
